In [ ]:
# Installation des dépendances (à exécuter une seule fois)
!pip install langchain langchain-community chromadb openai
!pip install -q langchain_huggingface sentence_transformers --no-deps
!pip install -U langchain-huggingface
!pip install -q bert-score


In [ ]:
# Imports globaux
import csv
import glob
import json
import random
import re
import shutil
import statistics
from datetime import datetime
from pathlib import Path

import torch
from bert_score import score as bert_score_fn
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


In [ ]:
# Configuration globale
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
RUN_CHAT = False
K = 4  # nombre de documents finaux retournés par le retrieval

CHUNK_SIZE = 400
CHUNK_OVERLAP = 100


EVAL_MAX_QUERIES = 120  # ex: 120 pour tests rapides, None pour toutes les requêtes
EVAL_RANDOM_SEED = 42

DATASET_FOLDER_PATH = Path("dataset_eval")
BERT_THRESHOLD = 0.60  # seuil accepté pour le F1 BERTScore (60%)
LANG = "fr"

CHROMA_DB_PATH = Path("./chroma_db")
CHROMA_EVAL_DB_PATH = Path("./chroma_eval_db")

RESULTS_FOLDER_PATH = DATASET_FOLDER_PATH / "result"
RESULTS_FOLDER_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_LAST_FOLDER_PATH = RESULTS_FOLDER_PATH / "last"
RESULTS_LAST_FOLDER_PATH.mkdir(parents=True, exist_ok=True)


In [ ]:
## 1) Préparation du retrieval - ingestion des PDF

# Chargement des PDF du dossier DIC
documents = []

# Chaque page devient un objet Document contenant :
# 	• 	page_content → le texte
# 	•	metadata → source, page, auteur, etc.

for file in glob.glob("DIC/*.pdf"):
    try:
        loader = PyPDFLoader(file)  # Retourne une liste de document (un pour chaque page)
        documents += loader.load()
    except Exception as e:
        print(f"Erreur survenue pour le fichier '{file}' : {e}")


In [ ]:
## 1.1) Vérification du chargement
# Première page du premier document PDF
documents[0]


Document(metadata={'producer': 'Actuate PDF Writer (Low Resolution) 2.1', 'creator': 'Actuate e.Reports', 'creationdate': '2022-07-29T08:11:45+01:00', 'title': '', 'subject': '', 'author': 'IDS GmbH - Analysis and Reporting Services', 'keywords': 'FR0010032326 (22.08.2022)', 'source': 'DIC/Allianz.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='\x00I\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00c\x00l\x00Ø\x00s\x00 \x00p\x00o\x00u\x00r\x00 \x00l \x19\x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\n\x00C\x00e\x00 \x00d\x00o\x00c\x00u\x00m\x00e\x00n\x00t\x00 \x00f\x00o\x00u\x00r\x00n\x00i\x00t\x00 \x00d\x00e\x00s\x00 \x00i\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00e\x00s\x00s\x00e\x00n\x00t\x00i\x00e\x00l\x00l\x00e\x00s\x00 \x00a\x00u\x00x\x00 \x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\x00s\x00 \x00d\x00e\x00 \x00c\x00e\x00t\x00 \x00O\x00P\x00C\x00V\x00M\x00.\x00 \x00I\x00l\x00 \x00n\x00e\x00 \x00s

In [ ]:
## 1.2) Découpage en chunks
# Initialisation du séparateur de texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,  # Taille maximale des morceaux de texte
    chunk_overlap=CHUNK_OVERLAP,  # Chevauchement entre les morceaux pour garder le contexte
    length_function=len,  # Fonction pour calculer la longueur des morceaux
    separators=["\n\n", "\n"]  # Séparateurs utilisés pour diviser le texte en morceaux
)

# Division du document en morceaux (chunks)
chunks = text_splitter.split_documents(documents=documents)

# Affichage du nombre de chunks créés
print(f"{len(chunks)} chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.")


1721 chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.


In [ ]:
## 1.3) Embeddings
# Choix du modèle d'embedding
# On normalise les embeddings pour améliorer la stabilité de la similarité cosinus.
# Modèle open-source multilingue adapté au français, exécutable localement sans API externe.
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, encode_kwargs={"normalize_embeddings" : True})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
## 1.4) Construction de la base vectorielle (Chroma)
# Nettoyage de la base `chroma_db` si elle existe

persist_path = CHROMA_DB_PATH

# Supprime récursivement le dossier persist_path et tout son contenu (fichiers + sous-dossiers)
if persist_path.exists():
    shutil.rmtree(persist_path)
persist_path.mkdir(parents=True, exist_ok=True)

# Créer la base Chroma (locale)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(persist_path),  # stockage local
    collection_name="col_production"
)


In [ ]:
## 1.5) Vérification rapide du retrieval
query = "Que montre le scénario de tensions ?"
results = vectorstore.similarity_search(query)

for r in results:
    print(r.page_content)


précision . Le scénario de tensions montre ce que vous pourriez obtenir dans des situations de marché extrêmes. 
Ce que vous obtiendrez de ce produit dépend des performances futures du marché. L’évolution future du marché est aléatoire et ne peut être prédite avec 
précision. 
Les chiﬀres indiqués comprennent tous les coûts du produit lui-même, mais pas 
nécessairement tous les frais dus à votre conseiller ou distributeur. Ces chiﬀres 
ne tiennent pas compte de votre situation ﬁscale personnelle, qui peut également 
inﬂuer sur les montants que vous recevrez.
Les scénarios défavorable, intermédiaire et favorable présentés représentent des exemples utilisant les meilleures et pires per formances, ainsi que la performance 
moyenne du produit au cours des 10 dernières années.  
Les marchés pourraient évoluer très différemment à l’avenir. Le scénario de tension montre ce que vous pourriez obtenir dans des situations de marché extrêmes. 
Il n'est pas facile de sortir de ce produit. Si vous s

In [ ]:
## 2) Initialisation du modèle de génération

# Modèle 7B choisi afin d’assurer un bon équilibre entre qualité de génération,
# stabilité GPU (24GB VRAM) et performance en temps de réponse,
# largement suffisant pour un RAG d’analyse factuelle de documents DIC.
# Dans une architecture RAG,
# la qualité dépend principalement du retrieval ; un modèle plus volumineux (14B)
# n’apporte pas de gain significatif pour l’analyse factuelle des DIC
# mais augmente fortement la consommation mémoire et le temps d’inférence.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)

print(model.config._name_or_path)

# 256 tokens maximum afin d’optimiser le temps d’inférence
# et prévenir les dépassements mémoire (OOM) observés avec des limites plus élevées.

#remettre max_new_tokens= 258 si pas evaluation
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    return_full_text=False
)

# Création d'une instance de HuggingFacePipeline à partir de l'identifiant du modèle spécifié (LangChain wrapper)
llm = HuggingFacePipeline(pipeline=generation_pipeline)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


mistralai/Mistral-7B-Instruct-v0.2


In [ ]:
## 2.1) Pipeline RAG (mémoire, prompt, génération)
memory = ConversationBufferMemory(
    memory_key="history",
    return_messages=False
)

prompt_template = """Tu es un assistant spécialisé en analyse de documents d’informations clés (DIC) pour un département ALM d’assurance vie.Réponds uniquement à partir du contexte fourni.
Tu ne dois jamais utiliser tes connaissances internes.
Réponds uniquement à partir du contexte fourni.

Tu dois obligatoirement citer la source exacte du document utilisé pour formuler la réponse.

Si aucune source n’est trouvée dans le contexte fourni,
indique explicitement :
"Information non disponible dans les DIC fournis."

Historique de la conversation :
{history}

Contexte :
{context}

Question :
{question}

Réponse (réponds uniquement à la question posée, sans ajouter de nouvelle question) :
"""

def has_valid_source_citation(text: str) -> bool:
    if not text or not text.strip():
        return False

    # Vérifie un format minimal de citation: "Source: ... - Page <nombre>"
    pattern = r"source\s*:\s*.+?\s*-\s*page\s*\d+"
    return re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL) is not None

def rag_pipeline(query, use_eval=False):
    # Choix du vectorstore
    current_vectorstore = vectorstore_eval if use_eval else vectorstore

     # Récupérer l'historique (PAS de memory en mode évaluation)
    if use_eval:
        history = ""
    else:
        history = memory.load_memory_variables({})["history"]
    
    # on recherche les documents
    # Limitation à K documents récupérés pour contrôler la longueur du prompt
    # et stabiliser le pipeline RAG.
    retrieved_docs = current_vectorstore.similarity_search(query, k=K)

    context="\n\n".join(
            f"Source: {doc.metadata.get('source', 'Inconnue')} - Page {doc.metadata.get('page', '?')}\n{doc.page_content}"
            for doc in retrieved_docs
    )
    
    # Ensuite, on injecte l'historique et les documents (avec leurs metadata) dans le prompt
    prompt = prompt_template.format(
        history=history,
        question=query,
        context=context
    )
    # Appel via llm (PAS pipeline direct)
    response = llm.invoke(prompt, stop=["Question:"])
    response_str = str(response)

    if not has_valid_source_citation(response_str):
        response_str += "\n\n[AVERTISSEMENT] Réponse non conforme: source/page manquante."

    # Sauvegarde dans la mémoire (sauf en mode evaluation)
    # 🔥 Ne pas sauvegarder en mode évaluation
    if not use_eval:
        memory.save_context({"input": query}, {"output": response_str})

    return response_str


/tmp/ipykernel_1722/3109492713.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [ ]:
## 2.2) Test d'une requête
query = """
Quel est l'objectif de gestion du FCP décrit dans le document ? ?
"""

# Effectuer une requête
response = rag_pipeline(query)
print(response)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



L'objectif de gestion du FCP décrit dans les documents est de mettre en œuvre une gestion discrétionnaire pour surperformer, via une exposition aux actions du secteur immobilier de l'Union européenne, la performance financière et extra-financière sur une durée de placement recommandée supérieure à 5 ans, en référence à l'indicateur FTSE EPRA / NAREIT Euro Zone Capped dividendes nets réinvestis après déduction des frais de gestion. Le F


In [ ]:
## 2.3) Boucle de chat interactive
def chat():
    print("Assistant DIC prêt. Tape 'exit' pour quitter.\n")
    
    while True:
        user_input = input("Vous : ")
        
        if user_input.lower() in ["exit", "quit"]:
            print("Fin de la conversation.")
            break
        
        response = rag_pipeline(user_input)
        print("\nAssistant :", response, "\n")


In [ ]:
## 2.4) Exécution conditionnelle du chat
memory.clear()
if RUN_CHAT:
    chat()


Assistant DIC prêt. Tape 'exit' pour quitter.



Vous :  exit


Fin de la conversation.


In [ ]:
## 3) Évaluation

# 3.1) Recréer la base de test à partir de `corpus.json`
# === Dépendances (installer si nécessaire) ===
# pip install chromadb sentence-transformers langchain tqdm

# Nettoyage de la base chroma_eval_db si elle existe
persist_path = CHROMA_EVAL_DB_PATH
if persist_path.exists():
    shutil.rmtree(persist_path)
persist_path.mkdir(parents=True, exist_ok=True)

# === 1️⃣ Charger le corpus ===
with open(DATASET_FOLDER_PATH / "corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Nombre de chunks : {len(corpus)}")

# === 2️⃣ Transformer en Documents ===
documents = [
    Document(page_content=text, metadata={"uuid": uid})
    for uid, text in corpus.items()
]

print(f"{len(documents)} Documents prêts.")

# === 3️⃣ Embeddings ===
# NOTE: Évaluation initiale faite avec sentence-transformers/all-MiniLM-L6-v2 (mode rapide).
# Pour une mesure fidèle de la prod, utiliser le même embedding que la prod:
# sentence-transformers/paraphrase-multilingual-mpnet-base-v2

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

# === 4️⃣ Vectorstore ===
vectorstore_eval = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=str(persist_path),
    collection_name="col_evaluation"
)

print("Base d’évaluation construite avec succès.")


In [ ]:
# 3.2) Évaluer le retrieval (Recall@k)
#      Évaluer la génération (F1 BERTScore vs `answers.json`)

# Réglages

# Charger fichiers
with open(DATASET_FOLDER_PATH / "queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)  # dict uuid -> query_text

with open(DATASET_FOLDER_PATH / "answers.json", "r", encoding="utf-8") as f:
    answers = json.load(f)  # dict uuid -> expected_answer_text

with open(DATASET_FOLDER_PATH / "relevant_docs.json", "r", encoding="utf-8") as f:
    relevant_docs = json.load(f)  # dict uuid -> [list of relevant doc uuids]

# Prépare stockage des résultats
results = []

# Boucle d'évaluation sur les UUID présents dans `queries`, `answers` et `relevant_docs`
common_uuids = sorted(
    set(queries.keys())
    .intersection(set(answers.keys()))
    .intersection(set(relevant_docs.keys()))
)

if EVAL_MAX_QUERIES is not None:
    rng = random.Random(EVAL_RANDOM_SEED)
    common_uuids = rng.sample(common_uuids, k=min(EVAL_MAX_QUERIES, len(common_uuids)))
    common_uuids = sorted(common_uuids)
print(f"Evaluation de {len(common_uuids)} requêtes (K={K}, max={EVAL_MAX_QUERIES}) — device torch: {torch.cuda.is_available() and 'cuda' or 'cpu'}")

# tqdm sert à visualiser la progression d’une boucle.
for uid in tqdm(common_uuids, desc="Evaluation RAG"):
    query = queries[uid].strip()
    ref_answer = answers[uid].strip()

    # 1) Générer la réponse via la pipeline d'évaluation
    try:
        raw_gen_answer = rag_pipeline(query, use_eval=True)
        # Si rag_pipeline renvoie un objet complexe, extraire string :
        if isinstance(raw_gen_answer, (list, tuple)):
            gen_answer = " ".join(str(x) for x in raw_gen_answer)
        else:
            gen_answer = str(raw_gen_answer)
    except Exception as e:
        gen_answer = ""
        print(f"[ERROR] génération pour {uid} a levé : {e}")

    # 2) retrieval (même comportement que dans la pipeline)
    retrieved_docs = vectorstore_eval.similarity_search(query, k=K)

    # Extraire identifiants/uuids des docs récupérés — on cherche un champ 'uuid' ou 'source' dans metadata
    # Chaque chunk indexé dans Chroma a son uuid du corpus.json.
    retrieved_uuids = [doc.metadata["uuid"] for doc in retrieved_docs]

    # 3) comparer aux relevant_docs attendus (liste)
    expected_uuids = relevant_docs[uid]
    # Le recall mesure la proportion des documents réellement pertinents
    # (définis dans relevant_docs.json) qui ont été correctement récupérés
    # par le moteur de recherche du pipeline RAG.
    if len(expected_uuids) == 0:
        recall_at_k = None
    else:
        found = sum(1 for e in expected_uuids if str(e) in retrieved_uuids)
        recall_at_k = found / len(expected_uuids)

    results.append({
        "uuid": uid,
        "query": query,
        "gen": gen_answer,
        "ref": ref_answer,
        "retrieved_uuids": retrieved_uuids,
        "expected_uuids": expected_uuids,
        "recall_at_k": recall_at_k,
        "has_source_citation": has_valid_source_citation(gen_answer),
    })

# 4) Calcul du BERTScore (en batch pour accélérer)
# Le BERTScore évalue la proximité sémantique entre deux textes à l’aide d’un modèle BERT.
print("Calcul du BERTScore (F1) — ceci peut prendre quelques secondes/minutes selon la GPU/CPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"

gen_answers = [r["gen"] for r in results]
ref_answers  = [r["ref"] for r in results]

# Le BERTScore utilisé dans cette évaluation correspond au F1,
# qui combine la précision et le rappel sémantiques entre la réponse générée et la référence.
P, R, F1 = bert_score_fn(cands=gen_answers,
                         refs=ref_answers,
                         lang=LANG,
                         model_type=None,  # laisser bert-score choisir le modèle adapté à la langue
                         device=device,
                         verbose=True)

# bert_score renvoie tensors — convertir en float
F1_list = [float(x) for x in F1]

# Remplir les résultats avec le F1
for i, uid in enumerate(common_uuids):
    results[i]["bert_f1"] = F1_list[i]

# 5) métriques agrégées
valid_f1s = [v for v in F1_list if not (v is None or (isinstance(v, float) and (v != v)))]
mean_f1 = statistics.mean(valid_f1s) if valid_f1s else 0.0
# %des query qui ont F1 > 60%
pct_above_threshold = 100.0 * sum(1 for v in valid_f1s if v >= BERT_THRESHOLD) / len(valid_f1s) if valid_f1s else 0.0

# calcul de la moyenne du recall
recalls = [r["recall_at_k"] for r in results if r["recall_at_k"] is not None]
mean_recall_at_k = statistics.mean(recalls) if recalls else None


Evaluation de 619 requêtes (K=2) — device torch: cuda


Evaluation RAG:   0%|          | 0/619 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluation RAG:   0%|          | 1/619 [00:03<38:20,  3.72s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluation RAG:   0%|          | 2/619 [00:06<33:40,  3.27s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Plea

Calcul du BERTScore (F1) — ceci peut prendre quelques secondes/minutes selon la GPU/CPU...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/15 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/10 [00:00<?, ?it/s]

done in 1.19 seconds, 519.61 sentences/sec


In [ ]:
## 3.3) Résultats agrégés et export principal
print("\n--- Résultats agrégés ---")
print(f"Nombre de requêtes: {len(common_uuids)}")
print(f"Mean BERT-F1 : {mean_f1:.4f}")
print(f"% exemples avec BERT-F1 >= {BERT_THRESHOLD*100:.0f}% : {pct_above_threshold:.2f}%")
print(f"Mean recall@{K} (sur les requêtes ayant des relevant_docs) : {mean_recall_at_k if mean_recall_at_k is not None else 'N/A'}")

# 6) Sauvegarder les résultats détaillés en CSV
OUT = RESULTS_LAST_FOLDER_PATH / "evaluation_results.csv"
with open(OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = ["uuid","query","gen","ref","bert_f1","retrieved_uuids","expected_uuids","recall_at_k","has_source_citation"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({
            "uuid": r["uuid"],
            "query": r["query"],
            "gen": r["gen"],
            "ref": r["ref"],
            "bert_f1": r.get("bert_f1"),
            "retrieved_uuids": "|".join(r["retrieved_uuids"]),
            "expected_uuids": "|".join([str(x) for x in r["expected_uuids"]]),
            "recall_at_k": r["recall_at_k"],
            "has_source_citation": r.get("has_source_citation")
        })

print(f"Résultats sauvegardés dans : {OUT}")

print("\n--- DEBUG RETRIEVAL (10 premiers exemples) ---")

for r in results[:10]:
    print("Expected:", r["expected_uuids"])
    print("Retrieved:", r["retrieved_uuids"])
    print()


# 7) Produire `errors.json` (cas problématiques)
ERRORS_OUT = RESULTS_LAST_FOLDER_PATH / "errors.json"
errors = [
    {
        "uuid": r["uuid"],
        "query": r["query"],
        "bert_f1": r.get("bert_f1"),
        "recall_at_k": r.get("recall_at_k"),
        "expected_uuids": r.get("expected_uuids"),
        "retrieved_uuids": r.get("retrieved_uuids"),
        "has_source_citation": r.get("has_source_citation"),
    }
    for r in results
    if (not r.get("gen", "").strip())
    or (r.get("bert_f1") is not None and r["bert_f1"] < BERT_THRESHOLD)
    or (r.get("recall_at_k") == 0)
    or (not r.get("has_source_citation", False))
]

with open(ERRORS_OUT, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"Errors sauvegardés dans : {ERRORS_OUT} ({len(errors)} cas)")



--- Résultats agrégés ---
Nombre de requêtes: 619
Mean BERT-F1 : 0.6869
% exemples avec BERT-F1 >= 60% : 83.52%
Mean recall@2 (sur les requêtes ayant des relevant_docs) : 0.27948303715670436
Résultats sauvegardés dans : dataset_eval/evaluation_results.csv

--- DEBUG RETRIEVAL (10 premiers exemples) ---
Expected: ['4927c2dd-78af-4878-9e69-6e231e19727a']
Retrieved: ['39555985-cbdb-45fa-840f-39c8b1330f3b', '43092c19-4ea0-48ac-bf63-3a952c0462f7']

Expected: ['f8f8cc85-cbbf-452e-9cab-e88ccea9dbec']
Retrieved: ['c8aff14e-0a6b-403e-96e5-cc517c9425e5', '6d85a9c6-f67e-44dd-a5d2-75699f879757']

Expected: ['88501310-936a-499e-8042-7a672cde47d4']
Retrieved: ['ca6b0d80-2944-4c32-b25e-19c7624e1673', '627b7200-6f0d-4190-855d-2e8b3c8f984d']

Expected: ['410cb5a0-89b4-4147-9180-2c05e39d41a8']
Retrieved: ['43322df9-9c87-4da0-abd7-eeb23d201b01', '410cb5a0-89b4-4147-9180-2c05e39d41a8']

Expected: ['a7b1d021-da96-4408-8b46-86b37ff72184']
Retrieved: ['a661416b-be2d-4141-8985-5fb6bfc902c0', '4927c2dd-78af-4

In [ ]:
## 3.4) Export versionné (timestamp)
# Nettoyer le nom du modèle pour qu'il soit compatible fichier
safe_model_name = re.sub(r"[^\w\-]", "_", MODEL_ID.split("/")[-1])

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

SUMMARY_OUT = RESULTS_LAST_FOLDER_PATH / f"evaluation_summary_{safe_model_name}_k{K}_thr{int(BERT_THRESHOLD*100)}_{timestamp}.csv"
DETAIL_OUT = RESULTS_LAST_FOLDER_PATH / f"evaluation_results_{safe_model_name}_k{K}_thr{int(BERT_THRESHOLD*100)}_{timestamp}.csv"

# 1️⃣ Sauvegarde détaillée
with open(DETAIL_OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = [
        "uuid","query","gen","ref","bert_f1",
        "retrieved_uuids","expected_uuids","recall_at_k","has_source_citation"
    ]
    
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for r in results:
        writer.writerow({
            "uuid": r["uuid"],
            "query": r["query"],
            "gen": r["gen"],
            "ref": r["ref"],
            "bert_f1": r.get("bert_f1"),
            "retrieved_uuids": "|".join(r["retrieved_uuids"]),
            "expected_uuids": "|".join([str(x) for x in r["expected_uuids"]]),
            "recall_at_k": r["recall_at_k"],
            "has_source_citation": r.get("has_source_citation")
        })

# 2️⃣ Sauvegarde résumé
with open(SUMMARY_OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = [
        "model_name",
        "k",
        "threshold",
        "nb_queries",
        "mean_bert_f1",
        "pct_above_threshold",
        f"mean_recall_at_{K}"
    ]
    
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    writer.writerow({
        "model_name": MODEL_ID,
        "k": K,
        "threshold": BERT_THRESHOLD,
        "nb_queries": len(common_uuids),
        "mean_bert_f1": round(mean_f1, 4),
        "pct_above_threshold": round(pct_above_threshold, 2),
        f"mean_recall_at_{K}": round(mean_recall_at_k, 4) if mean_recall_at_k is not None else "N/A"
    })

print(f"Détail sauvegardé dans : {DETAIL_OUT}")
print(f"Résumé sauvegardé dans : {SUMMARY_OUT}")


Détail sauvegardé dans : dataset_eval/evaluation_results_Mistral-7B-Instruct-v0_2_k2_thr60_20260310_140419.csv
Résumé sauvegardé dans : dataset_eval/evaluation_summary_Mistral-7B-Instruct-v0_2_k2_thr60_20260310_140419.csv
